In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

SVM

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha (L2 regularization)
alpha_values = [0.0001]

best_alpha = None
best_f1 = -1
best_svm_model = None

# Khối 1: Chuyển dữ liệu lên Numpy 
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đẩy sang Tensor CUDA
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
# Đối với Multi-class SVM, giữ nguyên nhãn số nguyên (0, 1, 2... 10)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Kiến trúc Linear SVM (Multi-class) bằng Pytorch
class TorchLinearSVM(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes) # Sửa output thành 11 classes
        
    def forward(self, x):
        return self.linear(x)

# PyTorch hỗ trợ sẵn MultiMarginLoss dùng để tối ưu Multi-class SVM
criterion = nn.MultiMarginLoss()

for alpha in alpha_values:
    model = TorchLinearSVM(input_dim, num_classes).to(device)
    
    # Khai báo Optimizer - sử dụng weight_decay thay thế L2 penalty
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=alpha)
    
    model.train()
    epochs = 100
    batch_size = 8192
    num_samples = X_train_t.shape[0]
    
    for epoch in range(epochs):
        indices = torch.randperm(num_samples, device=device)
        for i in range(0, num_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_X = X_train_t[idx]
            batch_y = y_train_t[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Tính toán trên GPU với Multi-Class Hinge Loss
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid_t)
        # Sử dụng argmax để lấy chỉ số có logit cao nhất (tương ứng với nhãn)
        y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
        
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

from sklearn.metrics import roc_auc_score

# Đánh giá trên test
print("\n--- Đánh giá Model Multi-class SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
best_svm_model.eval()
with torch.no_grad():
    test_outputs = best_svm_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()
    y_test_proba = torch.softmax(test_outputs, dim=1).cpu().numpy()

if num_classes == 2:
    auc_roc = roc_auc_score(y_test, y_test_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC: {auc_roc:.4f}")

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...
alpha = 0.000100 | Validation Macro F1: 0.7199

=> Quá trình huấn luyện và tuning hoàn tất trong 50.57 giây.
=> Cấu hình tốt nhất: alpha = 0.0001 với Validation Macro F1: 0.7199

--- Đánh giá Model Multi-class SVM Tốt nhất (alpha=0.0001) trên tập TEST ---
AUC-ROC: 0.9799
              precision    recall  f1-score   support

           0     0.7211    0.0679    0.1241     19846
           1     0.9985    1.0000    0.9993    484077
           2     0.5629    0.8004    0.6610      2515
           3     0.9421    0.9347    0.9384     36253
           4     0.4504    0.3638    0.4025     18979
           5     0.9170    0.9829    0.9488      2451
           6     0.7923    0.3825    0.5160     11847
           7     0.9996    0.9959    0.9977    107436
           8     0.4271    0.8314    0.5643     16746
           9     0.9937    0.6860    0.8117     41523
          10     

Random Forest

In [4]:
from xgboost import XGBRFClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [15]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    # Sử dụng XGBRFClassifier với cấu hình device='cuda' để chạy trên GPU
    rf_model = XGBRFClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

from sklearn.metrics import roc_auc_score

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
y_test_proba = best_rf_model.predict_proba(X_test)

num_classes = len(np.unique(y_train))
if num_classes == 2:
    auc_roc = roc_auc_score(y_test, y_test_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC: {auc_roc:.4f}")

print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [19:10:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


max_depth = 15 | Validation Macro F1: 0.8896

=> Quá trình huấn luyện và tuning hoàn tất trong 115.07 giây.
=> Cấu hình tốt nhất: max_depth = 15 với Validation Macro F1: 0.8896

--- Đánh giá mô hình Random Forest Tốt nhất (max_depth=15) trên tập TEST ---
AUC-ROC: 0.9337
              precision    recall  f1-score   support

           0     0.8210    0.8286    0.8248     19846
           1     0.9963    1.0000    0.9982    484077
           2     0.6987    0.9634    0.8100      2515
           3     0.9974    0.8714    0.9302     36253
           4     0.6948    0.8548    0.7665     18979
           5     0.9202    0.9931    0.9553      2451
           6     0.4137    0.7440    0.5318     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9448    0.9586    0.9517     16746
           9     1.0000    0.6904    0.8168     41523
          10     0.8069    0.8351    0.8207     18567

    accuracy                         0.9593    760240
   macro avg     0.8449  

Decision Tree

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử nghiệm các giá trị max_depth khác nhau
max_depth_values = [25]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng XGBClassifier với n_estimators=1 để thiết lập giống một Decision Tree chạy bằng GPU
    dt_model = XGBClassifier(
        n_estimators=1,
        learning_rate=1.0,
        max_depth=depth,
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

from sklearn.metrics import roc_auc_score

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
y_test_proba = best_dt_model.predict_proba(X_test)

num_classes = len(np.unique(y_train))
if num_classes == 2:
    auc_roc = roc_auc_score(y_test, y_test_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC: {auc_roc:.4f}")

print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 25 | Validation Macro F1: 0.8893

=> Quá trình huấn luyện và tuning hoàn tất trong 6.62 giây.
=> Cấu hình tốt nhất: max_depth = 25 với Validation Macro F1: 0.8893

--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth=25) trên tập TEST ---
AUC-ROC: 0.9269
              precision    recall  f1-score   support

           0     0.8158    0.8293    0.8225     19846
           1     0.9952    1.0000    0.9976    484077
           2     0.7098    0.9606    0.8164      2515
           3     0.9979    0.8718    0.9306     36253
           4     0.6787    0.8467    0.7535     18979
           5     0.9230    0.9829    0.9520      2451
           6     0.4684    0.8265    0.5980     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9438    0.9595    0.9516     16746
           9     1.0000    0.6872    0.8146     41523
          10     0.8175    0.8266    0.8

Logistic Regression

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpha và penalty l1, l2
alpha_values = [1e-6]
penalty_values = ['l1']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

# Khối 1: Chuyển đổi toàn bộ mảng dữ liệu về Numpy (nếu đang ở pandas DataFrame)
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đưa thẳng lên VRAM của GPU với PyTorch
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Định nghĩa mạng: 1 tầng Linear duy nhất + Hàm phạt Cross Entropy = Logistic Regression
class TorchLogisticRegression(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes)
        
    def forward(self, x):
        return self.linear(x)

for penalty in penalty_values:
    for alpha in alpha_values:
        model = TorchLogisticRegression(input_dim, num_classes).to(device)
        
        # Nếu l2 thì dùng tham số weight_decay tích hợp sẵn của Optimizer tự tính toán
        weight_decay = alpha if penalty == 'l2' else 0.0
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        epochs = 100
        batch_size = 8192
        num_samples = X_train_t.shape[0]
        
        for epoch in range(epochs):
            # Xáo trộn index bộ nhớ trực tiếp trên GPU để đảm bảo tốc độ tối đa
            indices = torch.randperm(num_samples, device=device)
            
            for i in range(0, num_samples, batch_size):
                idx = indices[i:i+batch_size]
                batch_X = X_train_t[idx]
                batch_y = y_train_t[idx]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Nếu l1 thì cộng tay penalty L1 Norm
                if penalty == 'l1':
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + alpha * l1_norm
                    
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            valid_outputs = model(X_valid_t)
            y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
            
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

from sklearn.metrics import roc_auc_score

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
best_logreg_model.eval()
with torch.no_grad():
    test_outputs = best_logreg_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()
    y_test_proba = torch.softmax(test_outputs, dim=1).cpu().numpy()

if num_classes == 2:
    auc_roc = roc_auc_score(y_test, y_test_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC: {auc_roc:.4f}")

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...
penalty = l1, alpha = 0.000001 | Validation Macro F1: 0.7147

=> Quá trình huấn luyện và tuning hoàn tất trong 589.56 giây.
=> Cấu hình tốt nhất: penalty = l1, alpha = 1e-06 với Validation Macro F1: 0.7147

--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty=l1, alpha=1e-06) trên tập TEST ---
AUC-ROC: 0.9854
              precision    recall  f1-score   support

           0     0.4631    0.1903    0.2698     19846
           1     0.9555    1.0000    0.9773    484077
           2     0.5795    0.7956    0.6706      2515
           3     0.9590    0.7571    0.8462     36253
           4     0.5110    0.3875    0.4408     18979
           5     0.9901    0.9772    0.9836      2451
           6     0.6470    0.6626    0.6547     11847
           7     1.0000    0.6593    0.7947    107436
           8     0.4278    0.8273    0.5639     16746
           9     

XGBoost

In [7]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import gc
import torch

print("\n--- BẮT ĐẦU HUẤN LUYỆN XGBOOST BASELINE ---")

# tính toán class weights cho XGBoost dựa trên phân bố lớp tập train
print("Đang tính toán Custom Smoothed Class Weights...")

unique_classes = np.unique(y_train)
raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}

# tinh chỉnh weight thủ công (dựa vào kết quả thực nghiệm)
if 4 in weights_dict: weights_dict[4] *= 0.6
if 6 in weights_dict: weights_dict[6] *= 1.5
if 9 in weights_dict: weights_dict[9] *= 2.5
if 1 in weights_dict: weights_dict[1] *= 0.8

# chuyển đổi thành mảng Sample Weight cho tập train
final_sample_weights = np.array([weights_dict[y] for y in y_train])

# train model xgboost
xgb_model = xgb.XGBClassifier(
    n_estimators=1000, # 1000 cây
    learning_rate=0.08,
    max_depth=5, # mỗi cây có độ sâu tối đa là 5
    min_child_weight=2,
    gamma=1.0,
    reg_lambda=2.0,
    reg_alpha=0.5,
    subsample=0.75,         
    colsample_bytree=0.7,
    max_delta_step=3,
    random_state=42,
    eval_metric='mlogloss',
    tree_method="hist",
    device="cuda",  
    early_stopping_rounds=40
)

# dọn VRAM
print("Đang dọn dẹp RAM & VRAM...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Đang huấn luyện XGBoost...")
# train bằng tập train, early stop bằng tập valid
try:
    xgb_model.fit(
        X_train, y_train,
        sample_weight=final_sample_weights,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=100 
    )
except Exception as e:
    print(f"Lỗi XGBoost chạy trên GPU, thử fallback về CPU... Lỗi: {e}")
    xgb_model.set_params(device='cpu')
    xgb_model.fit(
        X_train, y_train,
        sample_weight=final_sample_weights,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=100 
    )
    
print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")

# test trên tập test
print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

num_classes = len(np.unique(y_train))
if num_classes == 2:
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
    
print(f"Accuracy:            {acc*100:.2f}%")
print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%")
print(f"AUC-ROC:             {auc_roc * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))


--- BẮT ĐẦU HUẤN LUYỆN XGBOOST BASELINE ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56874	validation_1-mlogloss:1.56968
[100]	validation_0-mlogloss:0.04757	validation_1-mlogloss:0.13927
[110]	validation_0-mlogloss:0.04579	validation_1-mlogloss:0.14411
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 70

--- ĐÁNH GIÁ TRÊN TẬP TEST ---
Accuracy:            95.63%
F1-Score (Macro):    85.02%
F1-Score (Weighted): 95.89%
AUC-ROC:             99.67%

Classification Report:
              precision    recall  f1-score   support

           0     0.7038    0.8536    0.7715     19846
           1     0.9988    1.0000    0.9994    484077
           2     0.7503    0.9917    0.8543      2515
           3     0.9944    0.8703    0.9282     36253
           4     0.7438    0.7207    0.7321     18979
           5     0.9823    0.9939    0.9880      2451
           6     0.4169    0.8243    0.5537     1